# Question 9 
In this exercise, we will predict the number of applications received using the other variables in the College data set.

In [2]:
from ISLP import load_data
import numpy as np

College = load_data('College')
College

,Private,Apps,Accept,Enroll,Top10perc,Top25perc,F.Undergrad,P.Undergrad,Outstate,Room.Board,Books,Personal,PhD,Terminal,S.F.Ratio,perc.alumni,Expend,Grad.Rate
0,Yes,1660,1232,721,23,52,2885,537,7440,3300,450,2200,70,78,18.1,12,7041,60
1,Yes,2186,1924,512,16,29,2683,1227,12280,6450,750,1500,29,30,12.2,16,10527,56
2,Yes,1428,1097,336,22,50,1036,99,11250,3750,400,1165,53,66,12.9,30,8735,54
3,Yes,417,349,137,60,89,510,63,12960,5450,450,875,92,97,7.7,37,19016,59
4,Yes,193,146,55,16,44,249,869,7560,4120,800,1500,76,72,11.9,2,10922,15
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
772,No,2197,1515,543,4,26,3089,2029,6797,3900,500,1200,60,60,21.0,14,4469,40
773,Yes,1959,1805,695,24,47,2849,1107,11520,4960,600,1250,73,75,13.3,31,9189,83
774,Yes,2097,1915,695,34,61,2793,166,6900,4200,617,781,67,75,14.4,20,8323,49
775,Yes,10705,2453,1317,95,99,5217,83,19840,6510,630,2115,96,96,5.8,49,40386,99


### (a)  
Split the data set into a training set and a test set.

In [10]:
import pandas as pd
from sklearn.model_selection import train_test_split

College_d = pd.get_dummies(College, drop_first=True)

X = College_d.drop(columns=["Apps"])
y = College_d["Apps"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

(621, 17) (156, 17)


### (b)  
Fit a linear model using least squares on the training set, and report the test error obtained.

In [11]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

lm = LinearRegression()
lm.fit(X_train, y_train)

y_pred = lm.predict(X_test)
test_mse = mean_squared_error(y_test, y_pred)

print("Test MSE:", test_mse)


Test MSE: 1492443.3790390363


### (c)  
Fit a ridge regression model on the training set, with $\lambda$ chosen by cross-validation. Report the test error obtained.

In [13]:
from sklearn.linear_model import RidgeCV
from sklearn.metrics import mean_squared_error

lambdas = [0.1, 1, 10, 100, 200, 500, 1000]

ridge_cv = RidgeCV(alphas=lambdas)
ridge_cv.fit(X_train, y_train)

best_lambda = ridge_cv.alpha_
print("Best lambda:", best_lambda)

y_pred_ridge = ridge_cv.predict(X_test)
ridge_test_mse = mean_squared_error(y_test, y_pred_ridge)
print("Ridge Test MSE:", ridge_test_mse)


Best lambda: 10.0
Ridge Test MSE: 1478572.007302768


### (d)  
Fit a lasso model on the training set, with $\lambda$ chosen by cross-validation. Report the test error obtained, along with the number of non-zero coefficient estimates.

In [14]:
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error
import numpy as np

lambdas = np.logspace(-3, 3, 100)

lasso_cv = LassoCV(alphas=lambdas, cv=10, max_iter=5000)
lasso_cv.fit(X_train, y_train)

best_lambda = lasso_cv.alpha_
print("Best lambda:", best_lambda)

y_pred_lasso = lasso_cv.predict(X_test)
lasso_test_mse = mean_squared_error(y_test, y_pred_lasso)
print("Lasso Test MSE:", lasso_test_mse)

nonzero_coef_count = np.sum(lasso_cv.coef_ != 0)
print("Number of non-zero coefficients:", nonzero_coef_count)


Best lambda: 4.9770235643321135
Lasso Test MSE: 1484513.7116920394
Number of non-zero coefficients: 17


### (e)  
Fit a PCR model on the training set, with $M$ chosen by cross-validation. Report the test error obtained, along with the value of $M$ selected by cross-validation.

In [15]:
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import numpy as np

max_components = X_train.shape[1]
M_values = range(1, min(20, max_components) + 1)

kf = KFold(n_splits=10, shuffle=True, random_state=42)

cv_errors = []

for M in M_values:
    mse_list = []
    for train_idx, val_idx in kf.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        pca = PCA(n_components=M)
        X_tr_pca = pca.fit_transform(X_tr)
        X_val_pca = pca.transform(X_val)

        lm = LinearRegression()
        lm.fit(X_tr_pca, y_tr)

        preds = lm.predict(X_val_pca)
        mse_list.append(mean_squared_error(y_val, preds))

    cv_errors.append(np.mean(mse_list))

best_M = M_values[np.argmin(cv_errors)]
print("Best M:", best_M)

pca_final = PCA(n_components=best_M)
X_train_pca = pca_final.fit_transform(X_train)
X_test_pca = pca_final.transform(X_test)

lm_final = LinearRegression()
lm_final.fit(X_train_pca, y_train)

y_pred_pcr = lm_final.predict(X_test_pca)
pcr_test_mse = mean_squared_error(y_test, y_pred_pcr)

print("PCR Test MSE:", pcr_test_mse)


Best M: 17
PCR Test MSE: 1492443.3790390273
